# Course 2 · Week 4 — Solution: Decision Trees

## Setup — the cat dataset

Ten animals, three yes/no features each. The labels in `y` say which were cats. Your tree's job: figure out the right *sequence of questions* that lets it correctly classify each animal.

This is small enough to fit in your head and verify the tree by hand if you want.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Cat vs not-cat classification using three categorical features.
# 0/1 encoding so we can split with `==`.
# Feature 0: ear shape  (0 = floppy, 1 = pointy)
# Feature 1: face round (0 = no,     1 = yes)
# Feature 2: whiskers   (0 = no,     1 = yes)
X = np.array([
    [1, 1, 1],   # cat
    [0, 1, 1],   # cat
    [0, 0, 0],   # not cat
    [1, 0, 1],   # cat
    [1, 1, 1],   # cat
    [1, 1, 0],   # cat
    [0, 0, 0],   # not cat
    [1, 1, 0],   # cat
    [0, 1, 0],   # not cat
    [0, 1, 0],   # not cat
])
y = np.array([1, 1, 0, 1, 1, 1, 0, 1, 0, 0])

print(f"X shape {X.shape}")
print(f"Distribution: {(y==1).sum()} cats / {(y==0).sum()} not cats")


X shape (10, 3)
Distribution: 6 cats / 4 not cats


## Quick recap

A decision tree is a flowchart of yes/no questions. At each internal node, you ask one question; depending on the answer you go left or right. Leaves are the predictions.

The whole game is: **which question to ask first, and which next**. Algorithm: at every node, pick the feature whose split most reduces the *uncertainty* of the labels. We measure uncertainty with **entropy**, and reduction with **information gain**.

- Entropy of pure labels (all same class) = 0 (no uncertainty)
- Entropy of 50/50 labels = 1 (max uncertainty for binary)
- Info gain of a split = how much we reduced entropy by asking that question


## Exercise 1 — entropy

```python
def entropy(y):
    if len(y) == 0: return 0.0
    p = float(np.mean(y))
    if p == 0 or p == 1: return 0.0
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))
```

The pure-class short-circuit avoids `log(0)` (which is `-inf`). For the cat dataset (6 cats, 4 not cats), p = 0.6, entropy ≈ 0.971 — slightly less than 1.0 because the split is 60/40, not 50/50.


In [2]:
def entropy(y):
    if len(y) == 0:
        return 0.0
    p = float(np.mean(y))
    if p == 0 or p == 1:
        return 0.0
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))


print(f"entropy([1,1,1,1]) = {entropy(np.array([1,1,1,1])):.4f}")
print(f"entropy([1,0])     = {entropy(np.array([1,0])):.4f}")
print(f"entropy(y full)    = {entropy(y):.4f}")
assert abs(entropy(y) - 0.971) < 0.01
print("✓ entropy() works")


entropy([1,1,1,1]) = 0.0000
entropy([1,0])     = 1.0000
entropy(y full)    = 0.9710
✓ entropy() works


## Exercise 2 — info gain

```python
def info_gain(X, y, feature):
    parent = entropy(y)
    left_mask = X[:, feature] == 0
    n_left, n_right = int(left_mask.sum()), len(y) - int(left_mask.sum())
    if n_left == 0 or n_right == 0: return 0.0
    w_l, w_r = n_left/len(y), n_right/len(y)
    return parent - (w_l * entropy(y[left_mask]) + w_r * entropy(y[~left_mask]))
```

For our dataset:
- Feature 0 (ear shape): info gain ≈ 0.61 — **best split**, and intuitively right (pointy ears = strong cat signal)
- Feature 2 (whiskers): info gain ≈ 0.42 — also informative
- Feature 1 (face round): info gain ≈ 0.09 — weakest

The tree builder will pick feature 0 first.


In [3]:
def info_gain(X, y, feature):
    parent = entropy(y)
    left_mask = X[:, feature] == 0
    n_left = int(left_mask.sum())
    n_right = len(y) - n_left
    if n_left == 0 or n_right == 0:
        return 0.0
    w_left = n_left / len(y)
    w_right = n_right / len(y)
    return parent - (w_left * entropy(y[left_mask]) + w_right * entropy(y[~left_mask]))


gains = [info_gain(X, y, j) for j in range(3)]
print(f"feature 0 (ear shape):  info gain = {gains[0]:.4f}")
print(f"feature 1 (face round): info gain = {gains[1]:.4f}")
print(f"feature 2 (whiskers):   info gain = {gains[2]:.4f}")
print(f"\nBest feature: {int(np.argmax(gains))}")


feature 0 (ear shape):  info gain = 0.6100
feature 1 (face round): info gain = 0.0913
feature 2 (whiskers):   info gain = 0.4200

Best feature: 0


## Exercise 3 — tree builder

The full implementation handles:
- Empty subset → predict 0 (default)
- Pure subset → leaf with that class
- Hit max depth → leaf with majority class
- No useful split (best gain = 0) → leaf with majority class
- Otherwise: split on best feature, recurse

Training accuracy is 100% — with only 10 examples and 3 features, a tree at max-depth 4 has plenty of capacity to memorize. On bigger noisy data this would be overfitting; here it just means the dataset is fully separable.


In [4]:
class Node:
    def __init__(self, feature=None, left=None, right=None, prediction=None):
        self.feature = feature
        self.left = left
        self.right = right
        self.prediction = prediction


def build_tree(X, y, max_depth=4, depth=0):
    if len(y) == 0:
        return Node(prediction=0)
    if entropy(y) == 0 or depth >= max_depth:
        return Node(prediction=int(round(np.mean(y))))
    n_features = X.shape[1]
    gains = [info_gain(X, y, f) for f in range(n_features)]
    best = int(np.argmax(gains))
    if gains[best] == 0:
        return Node(prediction=int(round(np.mean(y))))
    left_mask = X[:, best] == 0
    return Node(
        feature=best,
        left=build_tree(X[left_mask], y[left_mask], max_depth, depth + 1),
        right=build_tree(X[~left_mask], y[~left_mask], max_depth, depth + 1),
    )


def predict_one(node, x):
    while node.prediction is None:
        node = node.left if x[node.feature] == 0 else node.right
    return node.prediction


def predict(tree, X):
    return np.array([predict_one(tree, x) for x in X])


tree = build_tree(X, y)
y_pred = predict(tree, X)
print(f"Training accuracy: {float((y_pred == y).mean()):.4f}")


Training accuracy: 1.0000


## Visualize the tree

A simple recursive printer with indentation. Each line is either "feature N? (0=left, 1=right)" or "→ predict 0/1".

You should see something like:
```
feature 0?
  left:        # not pointy ears
    feature 2?
      left: → predict 0   (no whiskers, no pointy ears = not cat)
      right: → predict 1
  right:       # pointy ears
    → predict 1   (always cat)
```


In [5]:
def show(node, depth=0):
    indent = "  " * depth
    if node.prediction is not None:
        print(f"{indent}→ predict {node.prediction}")
    else:
        print(f"{indent}feature {node.feature}? (0=left, 1=right)")
        print(f"{indent}left:")
        show(node.left, depth + 1)
        print(f"{indent}right:")
        show(node.right, depth + 1)


print("Trained decision tree:")
show(tree)


Trained decision tree:
feature 0? (0=left, 1=right)
left:
  feature 2? (0=left, 1=right)
  left:
    → predict 0
  right:
    → predict 1
right:
  → predict 1


## ⭐ Stretch — bagged forest

Bootstrap sampling is one line: `np.random.choice(n, n, replace=True)`. Train a tree on each bootstrap sample. Vote by majority at prediction time.

With our tiny dataset, the forest may actually under-perform a single tree — small bootstrap samples can miss critical examples. The intuition: bagging trades a small accuracy hit on training for **big robustness gains on real-world test data** (where overfitting on training would hurt). With 10,000 rows instead of 10, the forest would dominate.


In [6]:
np.random.seed(0)
n_trees = 5
trees = []
for _ in range(n_trees):
    idx = np.random.choice(len(X), len(X), replace=True)
    trees.append(build_tree(X[idx], y[idx]))

votes = np.array([predict(t, X) for t in trees])
forest_pred = (votes.mean(axis=0) >= 0.5).astype(int)
print(f"Bagged forest accuracy: {float((forest_pred == y).mean()):.4f}")
print("\nNote: on this tiny (10-row) dataset, a single tree already memorizes everything.")
print("Forests pay off most on noisy datasets where any single tree would overfit.")


Bagged forest accuracy: 0.9000

Note: on this tiny (10-row) dataset, a single tree already memorizes everything.
Forests pay off most on noisy datasets where any single tree would overfit.


## Wrap-up

Decision trees are simpler than neural networks, but they're a top-tier tool for tabular data. XGBoost (the same idea + boosting) wins more Kaggle competitions on tabular datasets than any neural-network approach.

What you didn't do: handle continuous features (you'd sort and try every midpoint as a threshold), regression trees (replace entropy with variance), or boosting (sequential trees that focus on prior errors). All extensions of the same `build_tree` skeleton.
